# PCGT Colab Runner (Drive Data, No Internet Download)

This notebook is cleaned for a simple workflow:

1. Clone or update the repo.
2. Install Python dependencies only (no dataset downloads).
3. Mount Google Drive once.
4. Use dataset directly from Drive via symlink.
5. Verify GPU.
6. Run either a quick test or all 7 datasets.

Assumption: your dataset is already present at `/content/drive/MyDrive/PCGT/data` with the same structure as local `data/`.

## 1) Clone Or Update Repository

Run this once per session to ensure code is current.

In [23]:
import os

if not os.path.exists('/content/PCGT'):
    !git clone https://github.com/ranjanchoubey/PCGT.git /content/PCGT
else:
    %cd /content/PCGT
    !git pull
    %cd /content

%cd /content/PCGT

/content/PCGT
Already up to date.
/content
/content/PCGT


## 2) Install Dependencies

This installs code dependencies only. It does not download datasets.

In [24]:
%cd /content/PCGT
!pip install -q -r requirements.txt
!pip install -q pymetis

/content/PCGT


## 3) Mount Drive And Use Data In-Place

This mounts Drive and creates `/content/PCGT/data` as a symlink to your Drive dataset.

No data copy is performed.

In [31]:
import os
from datetime import datetime
from google.colab import drive

# 1) Mount drive once
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=False)
else:
    print('Drive already mounted.')

# 2) Your uploaded folder in Drive
DRIVE_UPLOAD_ROOT = '/content/drive/MyDrive/PCGT_datasets'

# Supports either:
# - /content/drive/MyDrive/PCGT_datasets/data/...
# - /content/drive/MyDrive/PCGT_datasets/... (data files directly inside)
if os.path.isdir(os.path.join(DRIVE_UPLOAD_ROOT, 'data')):
    DRIVE_DATA_ROOT = os.path.join(DRIVE_UPLOAD_ROOT, 'data')
else:
    DRIVE_DATA_ROOT = DRIVE_UPLOAD_ROOT

if not os.path.isdir(DRIVE_DATA_ROOT):
    print('Could not find dataset root at:', DRIVE_DATA_ROOT)
    print('Top-level folders in MyDrive:')
    !ls -la /content/drive/MyDrive | head -n 50
    raise RuntimeError('Fix DRIVE_UPLOAD_ROOT and rerun this cell.')

LOCAL_DATA_ROOT = '/content/PCGT/data'
FORCE_DRIVE_LINK = True

# 3) Point /content/PCGT/data to Drive data (no copy)
if os.path.islink(LOCAL_DATA_ROOT):
    current = os.path.realpath(LOCAL_DATA_ROOT)
    if current != DRIVE_DATA_ROOT:
        os.remove(LOCAL_DATA_ROOT)
        os.symlink(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
elif not os.path.exists(LOCAL_DATA_ROOT):
    os.symlink(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
elif os.path.isdir(LOCAL_DATA_ROOT):
    if FORCE_DRIVE_LINK:
        backup_dir = f"{LOCAL_DATA_ROOT}_local_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        os.rename(LOCAL_DATA_ROOT, backup_dir)
        os.symlink(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
        print('Moved existing local directory to:', backup_dir)
    else:
        print(f'Using existing directory at {LOCAL_DATA_ROOT} (not a symlink).')
        print('Set FORCE_DRIVE_LINK=True to replace it with a Drive symlink.')
else:
    raise RuntimeError(f'Path exists and is not a directory/symlink: {LOCAL_DATA_ROOT}')

print('Drive upload root:', DRIVE_UPLOAD_ROOT)
print('Drive data root  :', DRIVE_DATA_ROOT)
print('Local data path  :', LOCAL_DATA_ROOT, '->', os.path.realpath(LOCAL_DATA_ROOT))

!ls -la /content/PCGT/data | head -n 20

Drive already mounted.
Moved existing local directory to: /content/PCGT/data_local_backup_20260322_080507
Drive upload root: /content/drive/MyDrive/PCGT_datasets
Drive data root  : /content/drive/MyDrive/PCGT_datasets/data
Local data path  : /content/PCGT/data -> /content/drive/MyDrive/PCGT_datasets/data
lrwxrwxrwx 1 root root 41 Mar 22 08:05 /content/PCGT/data -> /content/drive/MyDrive/PCGT_datasets/data


## 5) Verify GPU

Run this before training.

In [33]:
import torch
print('torch:', torch.__version__)
print('cuda :', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi | head -n 15

torch: 2.10.0+cu128
cuda : 12.8
cuda available: True
GPU: Tesla T4
Sun Mar 22 08:06:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |     

## 6) Quick Smoke Test (Cora, 1 Run)

Use this first to verify end-to-end training without waiting for all datasets.

In [34]:
%cd /content/PCGT/medium

DATA_DIR = '/content/PCGT/data/'  # keep trailing slash
!python -B main.py \
  --method pcgt --dataset cora \
  --lr 0.01 --num_layers 4 --hidden_channels 64 \
  --ours_layers 1 --num_reps 4 --partition_method metis \
  --use_graph --use_residual --backbone gcn \
  --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 \
  --no_feat_norm --seed 123 --device 0 \
  --epochs 500 --patience 200 --runs 1 --display_step 100 \
  --aggregate add --data_dir {DATA_DIR} \
  --num_partitions 7 --graph_weight 0.8 \
  --dropout 0.4 --weight_decay 5e-4 \
  --ours_weight_decay 0.001 --ours_dropout 0.2

/content/PCGT/medium
Namespace(data_dir='/content/PCGT/data/', dataset='cora', device=0, seed=123, cpu=False, epochs=500, runs=1, train_prop=0.5, valid_prop=0.25, protocol='semi', rand_split=False, rand_split_class=True, label_num_per_class=20, valid_num=500, test_num=1000, directed=False, method='pcgt', hidden_channels=64, num_layers=4, num_heads=1, alpha=0.5, use_bn=False, use_residual=True, use_graph=True, use_weight=False, use_init=False, use_act=False, attention='gcn', lr=0.01, weight_decay=0.0005, dropout=0.4, display_step=100, no_feat_norm=True, patience=200, graph_weight=0.8, ours_weight_decay=0.001, ours_use_weight=False, ours_use_residual=False, ours_use_act=False, backbone='gcn', ours_layers=1, ours_dropout=0.2, aggregate='add', hops=1, gat_heads=8, out_heads=1, lamda=0.1, num_elayers=2, encoder_emdim=768, num_partitions=7, num_reps=4, partition_method='metis')
cora
Num nodes: 2708
train:torch.Size([140]), valid:torch.Size([500]), test:torch.Size([1000])
features shape=torch

## 7) Optional Full Run (All 7 Datasets)

Runs all final configs and saves logs to Drive.

In [ ]:
%cd /content/PCGT/medium

import os
from datetime import datetime

DATA_DIR = '/content/PCGT/data/'  # keep trailing slash
RESULTS_DIR = '/content/drive/MyDrive/PCGT/results_colab'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f"{RESULTS_DIR}/{RUN_TAG}"
os.makedirs(RUN_DIR, exist_ok=True)

print('Data dir   :', DATA_DIR)
print('Results dir:', RUN_DIR)

# Cora
!python -B main.py --method pcgt --dataset cora --lr 0.01 --num_layers 4 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 7 --graph_weight 0.8 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 2>&1 | tee {RUN_DIR}/cora_pcgt_gpu.txt

# CiteSeer
!python -B main.py --method pcgt --dataset citeseer --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 7 --graph_weight 0.7 --dropout 0.5 --weight_decay 0.01 --ours_weight_decay 0.02 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/citeseer_pcgt_gpu.txt

# PubMed
!python -B main.py --method pcgt --dataset pubmed --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 50 --graph_weight 0.8 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/pubmed_pcgt_gpu.txt

# Chameleon
!python -B main.py --method pcgt --dataset chameleon --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 10 --graph_weight 0.8 --dropout 0.5 --weight_decay 0.001 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/chameleon_pcgt_gpu.txt

# Film
!python -B main.py --method pcgt --dataset film --lr 0.05 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --seed 123 --device 0 --epochs 500 --patience 200 --runs 10 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 5 --graph_weight 0.5 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/film_pcgt_gpu.txt

# Squirrel
!python -B main.py --method pcgt --dataset squirrel --lr 0.01 --num_layers 4 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 10 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 10 --graph_weight 0.8 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/squirrel_pcgt_gpu.txt

# Deezer
!python -B main.py --method pcgt --dataset deezer-europe --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method kmeans --use_graph --use_residual --backbone gcn --rand_split --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 50 --graph_weight 0.5 --dropout 0.6 --weight_decay 5e-5 --ours_weight_decay 0.01 --ours_dropout 0.4 2>&1 | tee {RUN_DIR}/deezer_pcgt_gpu.txt

print('\nAll runs completed.')
print('Logs saved at:', RUN_DIR)
!for f in {RUN_DIR}/*_gpu.txt; do echo "\n===== $(basename $f) ====="; grep -E "^[0-9]+ runs:" "$f" | tail -n 1; done

/content/PCGT/medium
Data dir   : /content/PCGT/data/
Results dir: /content/drive/MyDrive/PCGT/results_colab/20260322_080654
Namespace(data_dir='/content/PCGT/data/', dataset='cora', device=0, seed=123, cpu=False, epochs=500, runs=5, train_prop=0.5, valid_prop=0.25, protocol='semi', rand_split=False, rand_split_class=True, label_num_per_class=20, valid_num=500, test_num=1000, directed=False, method='pcgt', hidden_channels=64, num_layers=4, num_heads=1, alpha=0.5, use_bn=False, use_residual=True, use_graph=True, use_weight=False, use_init=False, use_act=False, attention='gcn', lr=0.01, weight_decay=0.0005, dropout=0.4, display_step=100, no_feat_norm=True, patience=200, graph_weight=0.8, ours_weight_decay=0.001, ours_use_weight=False, ours_use_residual=False, ours_use_act=False, backbone='gcn', ours_layers=1, ours_dropout=0.2, aggregate='add', hops=1, gat_heads=8, out_heads=1, lamda=0.1, num_elayers=2, encoder_emdim=768, num_partitions=7, num_reps=4, partition_method='metis')
cora
Num no